[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/VaR_CEE_FM/blob/main/VaR_CEE_Backtesting/VaR_CEE_Backtesting.ipynb)

# VaR_CEE_Backtesting

Performs comprehensive VaR and ES backtesting using Kupiec unconditional coverage test,
Christoffersen conditional coverage test, Acerbi-Szekely Z2 test for ES, and Basel
traffic light classification. Evaluates all 10 models across all CEE series.

**Published in:** *Can Foundation Models Manage Risk? Zero-Shot VaR and ES Forecasting for CEE Markets*

**Author:** Daniel Traian Pele

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import chi2
from matplotlib.patches import Patch

%matplotlib inline

# ── Configuration ──────────────────────────────────────────
MARKETS = {
    "Romania":  {"index": "BET",   "fx": "EURRON"},
    "Poland":   {"index": "WIG20", "fx": "EURPLN"},
    "Czechia":  {"index": "PX",    "fx": "EURCZK"},
    "Hungary":  {"index": "BUX",   "fx": "EURHUF"},
    "Bulgaria": {"index": "SOFIX", "fx": "USDBGN"},
}
VAR_ALPHAS = [0.01, 0.025, 0.05]
ES_ALPHA = 0.025
ROLLING_WINDOW = 250

BASELINE_MODELS = ["HS", "GJR-GARCH", "ARIMA-Conformal", "LSTM-Conformal"]
FM_MODELS = ["Chronos-2", "TimesFM-2.5", "Moirai-2.0"]
FM_CONF_MODELS = ["Chronos-2-Conf", "TimesFM-2.5-Conf", "Moirai-2.0-Conf"]
ALL_MODELS = BASELINE_MODELS + FM_MODELS + FM_CONF_MODELS

def get_all_series():
    series = []
    for country, info in MARKETS.items():
        series.append((country, info["index"], f"{info['index']}_ret", "index"))
        series.append((country, info["fx"], f"{info['fx']}_ret", "fx"))
    return series

np.random.seed(42)

VAR_DIR = Path("var_forecasts")
BT_DIR = Path("backtesting")
VAR_DIR.mkdir(exist_ok=True)
BT_DIR.mkdir(exist_ok=True)

## Upload VaR Forecast CSVs

Upload ALL model forecast CSV files (up to 100 files). These include:
- `HS_BET_ret.csv`, `GJR-GARCH_BET_ret.csv`, etc.
- `Chronos-2_BET_ret.csv`, `TimesFM-2.5_BET_ret.csv`, `Moirai-2.0_BET_ret.csv`, etc.
- `Chronos-2-Conf_BET_ret.csv`, etc.

In [ ]:
from google.colab import files
print("Upload all VaR forecast CSV files:")
uploaded = files.upload()
for fname, content in uploaded.items():
    dest = VAR_DIR / fname
    with open(dest, "wb") as f:
        f.write(content)
print(f"\nUploaded {len(uploaded)} files to {VAR_DIR}")

In [ ]:
def kupiec_test(violations, n, alpha):
    """Kupiec (1995) Unconditional Coverage test."""
    p_hat = violations / n
    if p_hat == 0:
        p_hat = 1e-10
    if p_hat >= 1:
        p_hat = 1 - 1e-10
    try:
        lr = -2 * (n * np.log(1 - alpha) + violations * np.log(alpha)
                    - (n - violations) * np.log(1 - p_hat) - violations * np.log(p_hat))
        p_value = 1 - chi2.cdf(lr, df=1)
    except Exception:
        lr, p_value = np.nan, np.nan
    return lr, p_value


def christoffersen_test(hit_series, alpha):
    """Christoffersen (1998) Conditional Coverage test."""
    hits = np.array(hit_series, dtype=int)
    n00 = n01 = n10 = n11 = 0
    for i in range(1, len(hits)):
        if hits[i-1] == 0 and hits[i] == 0: n00 += 1
        elif hits[i-1] == 0 and hits[i] == 1: n01 += 1
        elif hits[i-1] == 1 and hits[i] == 0: n10 += 1
        elif hits[i-1] == 1 and hits[i] == 1: n11 += 1

    n_total = n00 + n01 + n10 + n11
    n_viol = n01 + n11
    lr_uc, _ = kupiec_test(n_viol, n_total, alpha)

    p01 = n01 / max(n00 + n01, 1)
    p11 = n11 / max(n10 + n11, 1)
    p = (n01 + n11) / max(n_total, 1)

    try:
        if 0 < p01 < 1 and 0 < p11 < 1 and 0 < p < 1:
            lr_ind = -2 * ((n00 + n10) * np.log(1 - p) + (n01 + n11) * np.log(p)
                           - n00 * np.log(1 - p01) - n01 * np.log(p01)
                           - n10 * np.log(1 - p11) - n11 * np.log(p11))
        else:
            lr_ind = 0
    except Exception:
        lr_ind = 0

    lr_cc = lr_uc + lr_ind
    p_value = 1 - chi2.cdf(lr_cc, df=2)
    return lr_cc, p_value


def acerbi_szekely_Z2(returns, var_forecasts, es_forecasts, alpha):
    """Acerbi-Szekely (2014) Z2 test for ES backtesting."""
    hits = (returns < var_forecasts).astype(float)
    n = len(returns)
    n_violations = hits.sum()
    if n_violations == 0:
        return np.nan, np.nan

    es_safe = np.where(np.abs(es_forecasts) < 1e-10,
                        np.sign(es_forecasts) * 1e-10, es_forecasts)
    Z2 = np.sum(hits * returns / es_safe) / (n * alpha) + 1

    n_boot = 1000
    z2_boot = []
    for _ in range(n_boot):
        idx = np.random.choice(n, size=n, replace=True)
        h_b = hits[idx]
        r_b = returns[idx]
        e_b = es_safe[idx]
        if h_b.sum() > 0:
            z2_b = np.sum(h_b * r_b / e_b) / (n * alpha) + 1
            z2_boot.append(z2_b)
    if len(z2_boot) > 0:
        p_value = np.mean(np.array(z2_boot) <= Z2)
    else:
        p_value = np.nan
    return Z2, p_value


def basel_traffic_light(n_exceptions, window=250, alpha=0.01):
    """Basel traffic light classification."""
    if alpha == 0.01:
        if n_exceptions <= 4: return "Green"
        elif n_exceptions <= 9: return "Yellow"
        else: return "Red"
    elif alpha == 0.025:
        if n_exceptions <= 9: return "Green"
        elif n_exceptions <= 14: return "Yellow"
        else: return "Red"
    elif alpha == 0.05:
        if n_exceptions <= 17: return "Green"
        elif n_exceptions <= 22: return "Yellow"
        else: return "Red"
    return "Unknown"

In [ ]:
def backtest_model_series(model, series_id):
    """Run all backtests for one model x series combination."""
    fname = VAR_DIR / f"{model}_{series_id}.csv"
    if not fname.exists():
        return None

    df = pd.read_csv(fname, parse_dates=["date"])

    var_cols = {}
    for alpha in VAR_ALPHAS:
        alpha_str = str(alpha).replace("0.", "")
        for col in df.columns:
            if f"var_{alpha}" in col or f"var_{alpha_str}" in col:
                var_cols[alpha] = col
                break

    es_col = None
    for col in df.columns:
        if "es_" in col.lower():
            es_col = col
            break

    results = []
    for alpha in VAR_ALPHAS:
        if alpha not in var_cols:
            continue
        vc = var_cols[alpha]
        valid = df.dropna(subset=["y_true", vc])
        if len(valid) < 50:
            continue

        y = valid["y_true"].values
        var_f = valid[vc].values
        violations = (y < var_f).astype(int)
        n_viol = violations.sum()
        n_total = len(violations)
        viol_rate = n_viol / n_total

        kup_stat, kup_p = kupiec_test(n_viol, n_total, alpha)
        chr_stat, chr_p = christoffersen_test(violations, alpha)

        if n_total >= 250:
            last_250_viol = violations[-250:].sum()
        else:
            last_250_viol = n_viol
        tl = basel_traffic_light(last_250_viol, min(n_total, 250), alpha)

        row = {
            "model": model, "series": series_id, "alpha": alpha,
            "n_obs": n_total, "n_violations": n_viol, "violation_rate": viol_rate,
            "expected_rate": alpha, "kupiec_stat": kup_stat, "kupiec_p": kup_p,
            "kupiec_pass": kup_p > 0.05 if not np.isnan(kup_p) else False,
            "christoffersen_stat": chr_stat, "christoffersen_p": chr_p,
            "christoffersen_pass": chr_p > 0.05 if not np.isnan(chr_p) else False,
            "basel_tl": tl, "last250_violations": last_250_viol,
        }

        if alpha == ES_ALPHA and es_col is not None:
            valid_es = df.dropna(subset=["y_true", vc, es_col])
            if len(valid_es) >= 50:
                y_es = valid_es["y_true"].values
                var_es = valid_es[vc].values
                es_f = valid_es[es_col].values
                z2, z2_p = acerbi_szekely_Z2(y_es, var_es, es_f, alpha)
                row["as_z2"] = z2
                row["as_p"] = z2_p
                row["as_pass"] = z2_p > 0.05 if not np.isnan(z2_p) else False

        results.append(row)
    return results

In [ ]:
print("=" * 60)
print("BACKTESTING")
print("=" * 60)

all_series = get_all_series()
all_results = []

for model in ALL_MODELS:
    print(f"\n--- {model} ---")
    for country, raw_id, series_id, stype in all_series:
        bt = backtest_model_series(model, series_id)
        if bt:
            all_results.extend(bt)
            for r in bt:
                alpha = r['alpha']
                tl = r['basel_tl']
                kp = r.get('kupiec_pass', '')
                print(f"  {series_id} a={alpha}: "
                      f"violations={r['n_violations']}/{r['n_obs']} "
                      f"({r['violation_rate']:.3f}), "
                      f"Kupiec={'PASS' if kp else 'FAIL'}, TL={tl}")

if all_results:
    df_bt = pd.DataFrame(all_results)
    bt_path = BT_DIR / "backtesting_results.csv"
    df_bt.to_csv(bt_path, index=False, float_format="%.6f")
    print(f"\nAll results saved to {bt_path}")

In [ ]:
if all_results:
    print("\n" + "=" * 60)
    print("TRAFFIC LIGHT DASHBOARD (alpha = 1%)")
    print("=" * 60)
    tl_1 = df_bt[df_bt['alpha'] == 0.01][['model', 'series', 'basel_tl']]
    if len(tl_1) > 0:
        pivot = tl_1.pivot(index='model', columns='series', values='basel_tl')
        print(pivot.to_string())

In [ ]:
if all_results:
    print("GREEN RATE PER MODEL (all alphas)")
    green = df_bt.groupby('model')['basel_tl'].apply(
        lambda x: (x == 'Green').mean()
    ).sort_values(ascending=False)
    print(green.to_string())

In [ ]:
if all_results:
    df_1 = df_bt[df_bt['alpha'] == 0.01]
    if len(df_1) > 0:
        avg_viol = df_1.groupby('model')['violation_rate'].mean().sort_values()
        fig, ax = plt.subplots(figsize=(10, 5))
        colors = ['steelblue' if m in BASELINE_MODELS else 'coral' for m in avg_viol.index]
        ax.barh(range(len(avg_viol)), avg_viol.values, color=colors, edgecolor='black', linewidth=0.5)
        ax.set_yticks(range(len(avg_viol)))
        ax.set_yticklabels(avg_viol.index)
        ax.axvline(x=0.01, color='red', linestyle='--', label='Expected (1%)')
        ax.set_xlabel('Average violation rate')
        ax.set_title('VaR 1% Average Violation Rates by Model')
        ax.legend(loc='lower right')
        plt.tight_layout()
        plt.show()

In [ ]:
import zipfile, io

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(BT_DIR.glob("*.csv")):
        zf.write(f, f.name)
zip_buffer.seek(0)
with open("backtesting_results.zip", "wb") as fout:
    fout.write(zip_buffer.read())
files.download("backtesting_results.zip")